# Task 3: Value of Robustification (Section 3.2)

This notebook implements **Task 3** from the paper: comparing nominal and robust
first-stage designs under three scenarios to quantify the benefit of explicitly
optimising for disruptions.

**Background:**
- The **nominal** solution solves eq. (6), ignoring disruptions entirely.
- The **robust** solution solves the two-stage minimax problem eq. (12) via
  Algorithm 1 (Column-and-Constraint Generation, C&CG).
- Given fixed first-stage decisions $x$ and a realised disruption $\varepsilon$,
  the **second-stage SOCP** (Corollary 1, eq. 15) optimises adaptive pricing
  and allocation to maximise operational profit.

**Metrics defined in this notebook:**

| Metric | Definition |
|--------|------------|
| **VOR** (Value of Robustification) | $\pi^{\rm rob}(\varepsilon^*_{\rm rob}) - \pi^{\rm nom}(\varepsilon^*_{\rm nom})$ under each solution's own worst-case $\varepsilon^*$ |
| **Pre-disruption cost of robustness** | $\pi^{\rm nom}(\varepsilon_0) - \pi^{\rm rob}(\varepsilon_0)$, the "peace-time" profit sacrifice |

where $\varepsilon_0$ denotes the no-disruption scenario ($\varepsilon_j = 1\ \forall j$)
and $\varepsilon^*$ denotes the adversarial worst-case disruption found by the
separation oracle (Corollary 2, eq. 17).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rcflp import (
    instancemaker,
    solve_nominal,
    solve_CCG,
    evaluate_second_stage,
    worst_case_disruption,
    no_disruption_scenario,
    sample_disruptions,
    compute_cost_breakdown,
    compute_risk_metrics,
)

## 1. Configuration

In [ ]:
# Main instance parameters
In, Jn, Rn = 10, 8, 2
V_SCALE, W  = 0.75, 10.0
GAMMA, Hn   = 2, 2
N_SAMPLES   = 50      # Monte Carlo samples for average-case
SEED        = 42
TOL         = 0.01
TIME_LIMIT  = 600
DATA_PATH   = '../dataset.xlsx'

# Sensitivity: vary Gamma
GAMMA_RANGE = [0, 1, 2, 3, 4]

## 2. Solve nominal and robust problems

We first solve the **nominal problem** (eq. 6), which maximises expected profit
ignoring any disruptions:

$$\max_{x \in \mathcal{X}}\ Q(x,\,\varepsilon_0) \tag{6}$$

We then solve the **robust problem** (eq. 12) using Algorithm 1
(Column-and-Constraint Generation):

$$\max_{x \in \mathcal{X}}\ \min_{\varepsilon \in \Xi(\Gamma)}\ Q(x,\,\varepsilon) \tag{12}$$

The C&CG algorithm iterates between a master problem and a separation oracle
until the optimality gap falls below `TOL`. The nominal solution $x^{\rm nom}$
is used as a warm-start for the robust solver.

In [ ]:
inst  = instancemaker(In, Jn, Rn, V_SCALE, W, data_path=DATA_PATH)
nom   = solve_nominal(inst)
x_nom = nom['x_jr']
print(f"Nominal profit: {nom['profit']:,.1f}  ({nom['runtime']:.1f}s)")

ccg   = solve_CCG(inst, GAMMA, Hn, x_init=x_nom, tol=TOL,
                  time_limit=TIME_LIMIT, verbose=True)
x_rob = ccg['x_jr']
print(f"Robust profit (LB): {ccg['profit_LB']:,.1f}  converged={ccg['converged']}  ({ccg['runtime']:.1f}s)")

## 3. First-stage solution comparison

Before comparing profits, we inspect which facilities are opened by each
solution and the resulting fixed-cost investment.  A robust design may open
more or larger facilities (higher capacity tier $r$) to hedge against
disruptions, incurring higher fixed costs in exchange for resilience.

In [ ]:
R = inst['R']
J = inst['J']
nom_open = [(j,r) for (j,r),v in x_nom.items() if v > 0.5]
rob_open = [(j,r) for (j,r),v in x_rob.items() if v > 0.5]
fixed_nom = sum(inst['fixed_cost'][j,r]*x_nom[j,r] for j in J for r in R)
fixed_rob = sum(inst['fixed_cost'][j,r]*x_rob[j,r] for j in J for r in R)
print(f"Nominal opens: {sorted(nom_open)}  fixed={fixed_nom:,.0f}")
print(f"Robust  opens: {sorted(rob_open)}  fixed={fixed_rob:,.0f}")
print(f"Solutions differ: {sorted(nom_open) != sorted(rob_open)}")

## 4. Pre-disruption comparison

We evaluate both solutions under the **no-disruption scenario**
$\varepsilon_0 = (\varepsilon_{j0}=1,\ \varepsilon_{jh}=0\ \forall h>0)$,
meaning all facilities operate at full capacity ($\varepsilon_j = 1\ \forall j$).

For each solution the second-stage SOCP (Corollary 1, eq. 15) is solved to
find the optimal adaptive prices and allocations.  Prices are recovered via
eq. (4): $p_i = v_i (1 - \sum_j y_{ij})$.

The **pre-disruption cost of robustness** measures how much profit the robust
solution sacrifices in normal operating conditions:
$$\text{Pre-disruption cost} = \pi^{\rm nom}(\varepsilon_0) - \pi^{\rm rob}(\varepsilon_0)$$

In [ ]:
eps0 = no_disruption_scenario(inst, Hn)
eval_nom_0 = evaluate_second_stage(inst, x_nom, eps0, Hn)
eval_rob_0 = evaluate_second_stage(inst, x_rob, eps0, Hn)
print(f"No disruption — Nominal profit: {eval_nom_0['profit']:,.1f}")
print(f"No disruption — Robust  profit: {eval_rob_0['profit']:,.1f}")
print(f"Pre-disruption cost of robustness: {eval_nom_0['profit']-eval_rob_0['profit']:,.1f}")

## 5. Worst-case disruption comparison

For each first-stage solution we find its **own worst-case disruption**
$\varepsilon^*$ by solving the adversary's problem (Corollary 2, eq. 17):

$$\varepsilon^*(x) = \arg\min_{\varepsilon \in \Xi(\Gamma)}\ Q(x, \varepsilon) \tag{17}$$

We then perform a **cross-evaluation**: each solution is also tested under the
other's worst-case disruption.  This reveals whether the nominal design is
catastrophically hurt by the disruption that the robust solution was designed
to withstand.

The **Value of Robustification (VOR)** is defined as:
$$\text{VOR} = \pi^{\rm rob}(\varepsilon^*_{\rm rob}) - \pi^{\rm nom}(\varepsilon^*_{\rm nom})$$

A positive VOR means the robust solution achieves higher guaranteed profit
under adversarial disruptions.

In [ ]:
eps_wc_nom, rc_wc_nom = worst_case_disruption(inst, x_nom, GAMMA, Hn)
eps_wc_rob, rc_wc_rob = worst_case_disruption(inst, x_rob, GAMMA, Hn)

# Each solution under its own worst-case
eval_nom_wc_nom = evaluate_second_stage(inst, x_nom, eps_wc_nom, Hn)
eval_rob_wc_rob = evaluate_second_stage(inst, x_rob, eps_wc_rob, Hn)

# Cross-evaluation
eval_nom_wc_rob = evaluate_second_stage(inst, x_nom, eps_wc_rob, Hn)
eval_rob_wc_nom = evaluate_second_stage(inst, x_rob, eps_wc_nom, Hn)

VOR = eval_rob_wc_rob['profit'] - eval_nom_wc_nom['profit']
print(f"Nominal profit under its own ε*: {eval_nom_wc_nom['profit']:,.1f}")
print(f"Robust  profit under its own ε*: {eval_rob_wc_rob['profit']:,.1f}")
print(f"Value of Robustification (VOR):  {VOR:,.1f}")
print()
print(f"Cross-eval — Nominal under rob's ε*: {eval_nom_wc_rob['profit']:,.1f}")
print(f"Cross-eval — Robust  under nom's ε*: {eval_rob_wc_nom['profit']:,.1f}")

## 6. Average-case comparison

Worst-case analysis gives a conservative bound.  To assess average performance
we draw $N$ random disruption scenarios $\varepsilon^{(k)} \in \Xi(\Gamma)$
and evaluate each solution under each scenario (Corollary 1, eq. 15).

The sampling procedure assigns random disruption levels $h_j \in H$ to
facilities sequentially, respecting the budget constraint
$\sum_j \sum_h \frac{h}{H-1}\,\varepsilon_{jh} \le \Gamma$ (paper eq. 11).

This Monte Carlo average approximates:
$$\bar{\pi} = \mathbb{E}_{\varepsilon \sim \Xi}[Q(x, \varepsilon)]$$

In [ ]:
scenarios = sample_disruptions(inst, GAMMA, Hn, N_SAMPLES, seed=SEED)

profits_nom_avg = []
profits_rob_avg = []
for eps in scenarios:
    r_nom = evaluate_second_stage(inst, x_nom, eps, Hn)
    r_rob = evaluate_second_stage(inst, x_rob, eps, Hn)
    profits_nom_avg.append(r_nom['profit'])
    profits_rob_avg.append(r_rob['profit'])

print(f"Average profit (N={N_SAMPLES}) — Nominal: {np.mean(profits_nom_avg):,.1f}  Robust: {np.mean(profits_rob_avg):,.1f}")

## 6.1 Risk metrics

Beyond the mean, we compare the two solutions on five risk-oriented measures
computed from the same `N_SAMPLES` per-scenario profits generated above.

| Metric | Definition |
|--------|------------|
| **Min profit** | $\min_k \pi^k$ — absolute worst realised outcome |
| **VaR 95 %** | 5th-percentile profit: 95 % of scenarios exceed this value |
| **CVaR 5 %** | Mean of the worst 5 % of scenarios (tail expectation) |
| **CVaR 10 %** | Mean of the worst 10 % of scenarios |
| **P(loss)** | Fraction of scenarios with $\pi^k < 0$ |
| **Mean regret** | $\mathbb{E}_k[\max(\pi^k_{\rm nom},\pi^k_{\rm rob}) - \pi^k]$ |
| **Max regret** | $\max_k[\max(\pi^k_{\rm nom},\pi^k_{\rm rob}) - \pi^k]$ |

Regret is measured against the **better solution in each scenario** — no
additional solves required.

In [ ]:
rm = compute_risk_metrics(profits_nom_avg, profits_rob_avg)
n_scen = rm["n_scenarios"]

metric_labels = {
    "mean_profit" : ("Mean profit",             True,  "{:,.1f}"),
    "min_profit"  : ("Min profit",              True,  "{:,.1f}"),
    "pct5_profit" : ("VaR 95 % (5th pct)",      True,  "{:,.1f}"),
    "cvar5"       : ("CVaR 5 %  (worst 5 %)",   True,  "{:,.1f}"),
    "cvar10"      : ("CVaR 10 % (worst 10 %)",  True,  "{:,.1f}"),
    "prob_loss"   : ("P(profit < 0)",            False, "{:.1%}"),
    "mean_regret" : ("Mean regret",             False, "{:,.1f}"),
    "max_regret"  : ("Max regret",              False, "{:,.1f}"),
}

rows = []
for key, (label, higher_better, fmt) in metric_labels.items():
    n_val = rm["nominal"][key]
    r_val = rm["robust"][key]
    if higher_better:
        winner = "Robust" if r_val > n_val else ("Nominal" if n_val > r_val else "Tie")
    else:
        winner = "Robust" if r_val < n_val else ("Nominal" if n_val < r_val else "Tie")
    rows.append({
        "Metric"  : label,
        "Nominal" : fmt.format(n_val),
        "Robust"  : fmt.format(r_val),
        "Better"  : winner,
    })

df_rm = pd.DataFrame(rows)

def _highlight(row):
    styles = [""] * len(row)
    cols = list(row.index)
    if row["Better"] == "Robust":
        styles[cols.index("Robust")]  = "background-color: #d4edda"
        styles[cols.index("Nominal")] = "background-color: #f8d7da"
    elif row["Better"] == "Nominal":
        styles[cols.index("Nominal")] = "background-color: #d4edda"
        styles[cols.index("Robust")]  = "background-color: #f8d7da"
    return styles

display(
    df_rm.style
    .apply(_highlight, axis=1)
    .set_caption(f"Risk metrics over {n_scen} sampled scenarios  |  Γ={GAMMA}")
)

# ── Distribution plot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f"Profit distribution over {n_scen} scenarios  (Γ={GAMMA})", fontsize=12)

p_nom = np.array(profits_nom_avg)
p_rob = np.array(profits_rob_avg)

# Histogram
ax = axes[0]
bins = np.linspace(min(p_nom.min(), p_rob.min()), max(p_nom.max(), p_rob.max()), 25)
ax.hist(p_nom, bins=bins, alpha=0.55, color="steelblue",  label="Nominal")
ax.hist(p_rob, bins=bins, alpha=0.55, color="darkorange", label="Robust")
ax.axvline(0, color="black", lw=1, ls="--", label="Break-even")
nom_var = rm["nominal"]["pct5_profit"]
rob_var = rm["robust"]["pct5_profit"]
ax.axvline(nom_var, color="steelblue",  lw=1.5, ls=":", label="VaR 95% nom")
ax.axvline(rob_var, color="darkorange", lw=1.5, ls=":", label="VaR 95% rob")
ax.set_xlabel("Profit")
ax.set_ylabel("Count")
ax.set_title("Histogram")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Empirical CDF
ax = axes[1]
for profits, color, label in [(p_nom, "steelblue", "Nominal"), (p_rob, "darkorange", "Robust")]:
    xs = np.sort(profits)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    ax.plot(xs, ys, color=color, lw=2, label=label)
ax.axvline(0,    color="black", lw=1,   ls="--", label="Break-even")
ax.axhline(0.05, color="grey",  lw=0.8, ls=":")
ax.axhline(0.10, color="grey",  lw=0.8, ls=":")
ax.set_xlabel("Profit")
ax.set_ylabel("Cumulative probability")
ax.set_title("Empirical CDF (grey lines: 5 % and 10 %)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Summary table

In [ ]:
summary_data = {
    'Nominal profit': [
        eval_nom_0['profit'],
        eval_nom_wc_nom['profit'],
        np.mean(profits_nom_avg),
    ],
    'Robust profit': [
        eval_rob_0['profit'],
        eval_rob_wc_rob['profit'],
        np.mean(profits_rob_avg),
    ],
}
summary_df = pd.DataFrame(
    summary_data,
    index=['No disruption', 'Worst-case (own ε*)', f'Average case (N={N_SAMPLES})'],
)
summary_df['Difference (Robust-Nominal)'] = (
    summary_df['Robust profit'] - summary_df['Nominal profit']
)

print("=" * 70)
print("Profit comparison: Nominal vs Robust")
print("=" * 70)
print(summary_df.to_string(float_format=lambda x: f"{x:,.1f}"))
print()
print(f"Value of Robustification (VOR): {VOR:,.1f}")
print(f"Pre-disruption cost of robustness: {eval_nom_0['profit']-eval_rob_0['profit']:,.1f}")

# ---- Cost breakdown under worst-case ----
print()
print("=" * 70)
print("Cost breakdown under worst-case disruption")
print("=" * 70)
bd_nom = eval_nom_wc_nom['breakdown']
bd_rob = eval_rob_wc_rob['breakdown']
breakdown_df = pd.DataFrame(
    {
        'Nominal': [
            bd_nom['fixed_cost'],
            bd_nom['transport'],
            bd_nom['revenue'],
            bd_nom['congestion'],
            bd_nom['profit'],
        ],
        'Robust': [
            bd_rob['fixed_cost'],
            bd_rob['transport'],
            bd_rob['revenue'],
            bd_rob['congestion'],
            bd_rob['profit'],
        ],
    },
    index=['Fixed cost', 'Transport cost', 'Revenue', 'Congestion cost', 'Profit'],
)
print(breakdown_df.to_string(float_format=lambda x: f"{x:,.1f}"))

## 8. Sensitivity analysis: VOR vs $\Gamma$, $v$, and $w$

We sweep over three dimensions simultaneously:

| Parameter | Meaning | Values |
|-----------|---------|--------|
| $\Gamma$ | Uncertainty budget | 0, 1, 2, 3, 4 |
| `v_scale` | Willingness-to-pay scale ($v_i = \text{v\_scale} \times d_{ij}^{\max}$) | 0.50, 0.75, 1.00 |
| `w` | Congestion cost weight | 1, 10, 100 |

For each combination we solve the nominal and robust (C&CG) problems, then
evaluate both solutions under three perspectives:

1. **No disruption** $\varepsilon_0$ — pre-disruption cost of robustness.
2. **Own worst-case** $\varepsilon^*(x)$ via eq. (17) — worst-case VOR.
3. **Out-of-sample (OOS) Monte Carlo** — `N_SAMPLES_SENS` scenarios drawn
   from $\Xi(\Gamma)$ via `sample_disruptions`.  Each evaluation is a convex
   SOCP (no integer variables), so this is fast even for large sample counts.
   Scenarios are generated once per $\Gamma$ and **reused** across all $(v, w)$
   combinations to make OOS results directly comparable.

All results are saved to `sensitivity_VOR.xlsx`.

In [ ]:
V_SCALE_VALUES  = [0.50, 0.75, 1.00]
W_VALUES        = [1.0, 10.0, 100.0]
N_SAMPLES_SENS  = 50      # OOS sample count — raise to 200 or 1000 if time allows
# GAMMA_RANGE and SEED are already defined in Section 1
SENS_TIME_LIMIT = 300     # per CCG solve in the sweep

total_runs = len(GAMMA_RANGE) * len(V_SCALE_VALUES) * len(W_VALUES)
print(
    f"Sensitivity sweep: {total_runs} combinations "
    f"({len(GAMMA_RANGE)} Γ × {len(V_SCALE_VALUES)} v × {len(W_VALUES)} w), "
    f"{N_SAMPLES_SENS} OOS samples each"
)
print()

# Pre-generate OOS scenarios once per Gamma — reused across all (v, w).
_inst_ref = instancemaker(In, Jn, Rn, V_SCALE_VALUES[0], W_VALUES[0], data_path=DATA_PATH)
oos_scenarios = {}
for gamma in GAMMA_RANGE:
    if gamma == 0:
        oos_scenarios[0] = [no_disruption_scenario(_inst_ref, Hn)]
    else:
        oos_scenarios[gamma] = sample_disruptions(
            _inst_ref, gamma, Hn, N_SAMPLES_SENS, seed=SEED
        )
print("OOS scenarios pre-generated: "
      + ", ".join(f"Γ={g}: {len(s)}" for g, s in oos_scenarios.items()))
print()

sens_rows = []

for v in V_SCALE_VALUES:
    for w in W_VALUES:
        print(f"=== v={v}  w={w} ===")
        inst_sw  = instancemaker(In, Jn, Rn, v, w, data_path=DATA_PATH)
        nom_sw   = solve_nominal(inst_sw)
        x_nom_sw = nom_sw["x_jr"]
        eps0_sw  = no_disruption_scenario(inst_sw, Hn)
        nom_profit = nom_sw["profit"]
        print(f"  Nominal profit: {nom_profit:,.1f}")

        for gamma in GAMMA_RANGE:

            # ── Solve robust (reuse nominal solution for Γ=0) ─────────
            if gamma == 0:
                x_rob_sw      = x_nom_sw
                rob_profit_lb = nom_sw["profit"]
                rob_converged = True
                rob_iters     = 0
                rob_runtime   = 0.0
            else:
                ccg_sw = solve_CCG(
                    inst_sw, gamma, Hn,
                    x_init=x_nom_sw,
                    tol=TOL,
                    time_limit=SENS_TIME_LIMIT,
                    verbose=False,
                )
                x_rob_sw      = ccg_sw["x_jr"]
                rob_profit_lb = ccg_sw["profit_LB"]
                rob_converged = ccg_sw["converged"]
                rob_iters     = ccg_sw["n_iter"]
                rob_runtime   = ccg_sw["runtime"]

            # ── No-disruption evaluation ──────────────────────────────
            e_nom_nd = evaluate_second_stage(inst_sw, x_nom_sw, eps0_sw, Hn)
            e_rob_nd = evaluate_second_stage(inst_sw, x_rob_sw, eps0_sw, Hn)
            pre_cost = e_nom_nd["profit"] - e_rob_nd["profit"]

            # ── Worst-case evaluation ─────────────────────────────────
            if gamma == 0:
                e_nom_wc = e_nom_nd
                e_rob_wc = e_rob_nd
            else:
                eps_wc_nom_sw, _ = worst_case_disruption(inst_sw, x_nom_sw, gamma, Hn)
                eps_wc_rob_sw, _ = worst_case_disruption(inst_sw, x_rob_sw, gamma, Hn)
                e_nom_wc = evaluate_second_stage(inst_sw, x_nom_sw, eps_wc_nom_sw, Hn)
                e_rob_wc = evaluate_second_stage(inst_sw, x_rob_sw, eps_wc_rob_sw, Hn)

            vor_wc = e_rob_wc["profit"] - e_nom_wc["profit"]

            # ── OOS Monte Carlo (SOCP only — no MIP, fast per call) ──
            oos_nom_profits, oos_rob_profits = [], []
            for eps_s in oos_scenarios[gamma]:
                oos_nom_profits.append(
                    evaluate_second_stage(inst_sw, x_nom_sw, eps_s, Hn)["profit"]
                )
                oos_rob_profits.append(
                    evaluate_second_stage(inst_sw, x_rob_sw, eps_s, Hn)["profit"]
                )

            nom_avg = np.mean(oos_nom_profits)
            rob_avg = np.mean(oos_rob_profits)
            vor_avg = rob_avg - nom_avg

            # ── Risk metrics ──────────────────────────────────────────
            rm_sw  = compute_risk_metrics(oos_nom_profits, oos_rob_profits)
            rn     = rm_sw["nominal"]
            rr     = rm_sw["robust"]

            print(f"  Γ={gamma}  VOR_wc={vor_wc:,.1f}  VOR_avg={vor_avg:,.1f}"
                  f"  pre_cost={pre_cost:,.1f}"
                  f"  nom_cvar5={rn['cvar5']:,.1f}  rob_cvar5={rr['cvar5']:,.1f}"
                  f"  converged={rob_converged}  ({rob_runtime:.0f}s)")

            sens_rows.append({
                "v_scale":              v,
                "w":                    w,
                "Gamma":                gamma,
                # no-disruption
                "nominal_profit_nd":    e_nom_nd["profit"],
                "robust_profit_nd":     e_rob_nd["profit"],
                "pre_disruption_cost":  pre_cost,
                # worst-case
                "nominal_profit_wc":    e_nom_wc["profit"],
                "robust_profit_wc":     e_rob_wc["profit"],
                "VOR_wc":               vor_wc,
                # OOS average
                "nominal_profit_avg":   nom_avg,
                "nominal_profit_std":   np.std(oos_nom_profits),
                "robust_profit_avg":    rob_avg,
                "robust_profit_std":    np.std(oos_rob_profits),
                "VOR_avg":              vor_avg,
                "n_oos_samples":        len(oos_scenarios[gamma]),
                # risk metrics — nominal
                "nom_min_profit":       rn["min_profit"],
                "nom_pct5":             rn["pct5_profit"],
                "nom_cvar5":            rn["cvar5"],
                "nom_cvar10":           rn["cvar10"],
                "nom_prob_loss":        rn["prob_loss"],
                "nom_mean_regret":      rn["mean_regret"],
                "nom_max_regret":       rn["max_regret"],
                # risk metrics — robust
                "rob_min_profit":       rr["min_profit"],
                "rob_pct5":             rr["pct5_profit"],
                "rob_cvar5":            rr["cvar5"],
                "rob_cvar10":           rr["cvar10"],
                "rob_prob_loss":        rr["prob_loss"],
                "rob_mean_regret":      rr["mean_regret"],
                "rob_max_regret":       rr["max_regret"],
                # solver info
                "rob_profit_lb":        rob_profit_lb,
                "rob_converged":        rob_converged,
                "rob_iters":            rob_iters,
                "rob_runtime_s":        round(rob_runtime, 1),
            })

sens_df = pd.DataFrame(sens_rows)
print()
print(f"Done. {len(sens_df)} rows collected.")

In [ ]:
EXCEL_PATH = "sensitivity_VOR.xlsx"

col_order = [
    "v_scale", "w", "Gamma",
    # no-disruption
    "nominal_profit_nd",   "robust_profit_nd",   "pre_disruption_cost",
    # worst-case
    "nominal_profit_wc",   "robust_profit_wc",   "VOR_wc",
    # OOS average
    "nominal_profit_avg",  "nominal_profit_std",
    "robust_profit_avg",   "robust_profit_std",  "VOR_avg",
    "n_oos_samples",
    # risk metrics — nominal
    "nom_min_profit",  "nom_pct5",  "nom_cvar5",  "nom_cvar10",
    "nom_prob_loss",   "nom_mean_regret",  "nom_max_regret",
    # risk metrics — robust
    "rob_min_profit",  "rob_pct5",  "rob_cvar5",  "rob_cvar10",
    "rob_prob_loss",   "rob_mean_regret",  "rob_max_regret",
    # solver info
    "rob_profit_lb", "rob_converged", "rob_iters", "rob_runtime_s",
]
df_out = sens_df[col_order].copy()

def _pivot(col):
    return sens_df.pivot_table(index="Gamma", columns=["v_scale", "w"], values=col)

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    # Sheet 1 — full flat table (all columns)
    df_out.to_excel(writer, sheet_name="results", index=False, float_format="%.4f")

    # VOR sheets
    _pivot("VOR_wc").to_excel(writer, sheet_name="VOR_wc",              float_format="%.2f")
    _pivot("pre_disruption_cost").to_excel(writer, sheet_name="pre_disruption_cost", float_format="%.2f")
    _pivot("VOR_avg").to_excel(writer, sheet_name="VOR_avg",            float_format="%.2f")

    # Raw profit sheets
    _pivot("nominal_profit_avg").to_excel(writer, sheet_name="nominal_profit_avg", float_format="%.2f")
    _pivot("robust_profit_avg").to_excel( writer, sheet_name="robust_profit_avg",  float_format="%.2f")
    _pivot("nominal_profit_wc").to_excel( writer, sheet_name="nominal_profit_wc",  float_format="%.2f")
    _pivot("robust_profit_wc").to_excel(  writer, sheet_name="robust_profit_wc",   float_format="%.2f")

    # Risk metric sheets — CVaR 5%
    _pivot("nom_cvar5").to_excel( writer, sheet_name="nom_cvar5",  float_format="%.2f")
    _pivot("rob_cvar5").to_excel( writer, sheet_name="rob_cvar5",  float_format="%.2f")

    # Risk metric sheets — CVaR 10%
    _pivot("nom_cvar10").to_excel(writer, sheet_name="nom_cvar10", float_format="%.2f")
    _pivot("rob_cvar10").to_excel(writer, sheet_name="rob_cvar10", float_format="%.2f")

    # Risk metric sheets — VaR 95% (5th pct)
    _pivot("nom_pct5").to_excel(  writer, sheet_name="nom_pct5",   float_format="%.2f")
    _pivot("rob_pct5").to_excel(  writer, sheet_name="rob_pct5",   float_format="%.2f")

    # Risk metric sheets — min profit
    _pivot("nom_min_profit").to_excel(writer, sheet_name="nom_min_profit", float_format="%.2f")
    _pivot("rob_min_profit").to_excel(writer, sheet_name="rob_min_profit", float_format="%.2f")

    # Risk metric sheets — P(loss)
    _pivot("nom_prob_loss").to_excel(writer, sheet_name="nom_prob_loss", float_format="%.4f")
    _pivot("rob_prob_loss").to_excel(writer, sheet_name="rob_prob_loss", float_format="%.4f")

    # Risk metric sheets — mean regret
    _pivot("nom_mean_regret").to_excel(writer, sheet_name="nom_mean_regret", float_format="%.2f")
    _pivot("rob_mean_regret").to_excel(writer, sheet_name="rob_mean_regret", float_format="%.2f")

    # Risk metric sheets — max regret
    _pivot("nom_max_regret").to_excel(writer, sheet_name="nom_max_regret", float_format="%.2f")
    _pivot("rob_max_regret").to_excel(writer, sheet_name="rob_max_regret", float_format="%.2f")

print(f"Saved: {EXCEL_PATH}  ({len(df_out)} rows, {len(df_out.columns)} columns)")
print()
print("VOR_wc pivot:")
print(_pivot("VOR_wc").to_string(float_format=lambda x: f"{x:,.1f}"))
print()
print("CVaR 5% pivot (nominal):")
print(_pivot("nom_cvar5").to_string(float_format=lambda x: f"{x:,.1f}"))
print()
print("CVaR 5% pivot (robust):")
print(_pivot("rob_cvar5").to_string(float_format=lambda x: f"{x:,.1f}"))

## 9. Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ── Panel (0,0): profit comparison for the main instance ──────────────────
ax = axes[0, 0]
scenario_labels = ['No disruption', 'Worst-case', f'Average (N={N_SAMPLES})']
nom_vals = [eval_nom_0['profit'], eval_nom_wc_nom['profit'], np.mean(profits_nom_avg)]
rob_vals = [eval_rob_0['profit'], eval_rob_wc_rob['profit'], np.mean(profits_rob_avg)]

xp    = np.arange(len(scenario_labels))
width = 0.35
ax.bar(xp - width/2, nom_vals, width, label='Nominal',
       color='steelblue', alpha=0.85)
ax.bar(xp + width/2, rob_vals, width, label='Robust',
       color='darkorange', alpha=0.85)
ax.set_xticks(xp)
ax.set_xticklabels(scenario_labels, fontsize=9)
ax.set_ylabel('Profit')
ax.set_title(f'Main instance (v={V_SCALE}, w={W}, \u0393={GAMMA})', fontsize=10)
ax.legend(fontsize=9)
ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Panel (0,1): VOR_wc vs Γ — one line per v_scale (fixed w = W) ────────
ax = axes[0, 1]
sub = sens_df[sens_df['w'] == W]
for v_val, grp in sub.groupby('v_scale'):
    ax.plot(grp['Gamma'], grp['VOR_wc'], marker='o', linewidth=2,
            label=f'v={v_val}')
ax.axhline(0, color='black', linewidth=0.7, linestyle=':')
ax.set_xlabel('Uncertainty budget \u0393')
ax.set_ylabel('VOR (worst-case)')
ax.set_title(f'Worst-case VOR vs \u0393  (w={W}, varying v)', fontsize=10)
ax.set_xticks(GAMMA_RANGE)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Panel (1,0): VOR_wc vs Γ — one line per w (fixed v = V_SCALE) ─────────
ax = axes[1, 0]
sub = sens_df[sens_df['v_scale'] == V_SCALE]
for w_val, grp in sub.groupby('w'):
    ax.plot(grp['Gamma'], grp['VOR_wc'], marker='s', linewidth=2,
            label=f'w={w_val:.0f}')
ax.axhline(0, color='black', linewidth=0.7, linestyle=':')
ax.set_xlabel('Uncertainty budget \u0393')
ax.set_ylabel('VOR (worst-case)')
ax.set_title(f'Worst-case VOR vs \u0393  (v={V_SCALE}, varying w)', fontsize=10)
ax.set_xticks(GAMMA_RANGE)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# ── Panel (1,1): VOR_avg and pre-disruption cost vs Γ (v=V_SCALE, w=W) ───
ax = axes[1, 1]
sub = sens_df[(sens_df['v_scale'] == V_SCALE) & (sens_df['w'] == W)]
ax.plot(sub['Gamma'], sub['VOR_avg'], marker='^', linewidth=2,
        color='darkorange', label=f'VOR (avg, N={N_SAMPLES_SENS})')
ax.plot(sub['Gamma'], sub['VOR_wc'], marker='o', linewidth=2,
        color='firebrick', linestyle='--', label='VOR (worst-case)')
ax.plot(sub['Gamma'], sub['pre_disruption_cost'], marker='s', linewidth=2,
        color='steelblue', linestyle=':', label='Pre-disruption cost')
ax.axhline(0, color='black', linewidth=0.7, linestyle=':')
ax.set_xlabel('Uncertainty budget \u0393')
ax.set_ylabel('Profit difference')
ax.set_title(f'VOR (avg & WC) and pre-disruption cost  (v={V_SCALE}, w={W})', fontsize=10)
ax.set_xticks(GAMMA_RANGE)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

plt.tight_layout()
plt.savefig('fig_robustification.pdf', bbox_inches='tight')
plt.savefig('fig_robustification.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: fig_robustification.pdf / .png")

## 10. Interpretation

**Pre-disruption cost of robustness.**  The nominal solution is tuned to
maximise profit when no disruptions occur.  Because the robust solution
hedges by opening facilities with higher capacity (or at different locations)
it typically achieves slightly *lower* profit under $\varepsilon_0$.  This
"insurance premium" is the pre-disruption cost of robustness.

**Value of Robustification (VOR).**  Under the adversarial worst-case
disruption, the nominal design can suffer severe profit losses because it was
never designed to handle capacity reductions.  The robust design, by
construction (eq. 12), minimises this downside.  A positive VOR confirms that
the robust solution provides a meaningfully higher guaranteed profit floor.

**Sensitivity to $\Gamma$.**  As the uncertainty budget grows:
- The adversary can inflict heavier damage on the nominal solution, so the
  **nominal worst-case profit falls steeply**.
- The robust solution invests more in resilience, **increasing the
  pre-disruption cost** while maintaining a higher profit floor.
- The **VOR therefore increases with $\Gamma$**: the benefit of robustification
  is largest when disruptions can be severe.

**Average-case results.**  The Monte Carlo evaluation (Section 3.2) shows
that under typical (non-adversarial) disruptions the robust solution often
performs comparably to or better than the nominal solution — the pre-disruption
cost of robustness is small relative to the VOR.

Together these results justify the use of the robust formulation eq. (12)
whenever disruptions are possible: a modest sacrifice in best-case profit
yields a substantially higher guaranteed profit under adversarial conditions.